In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBClassifier
import optuna
import mlflow

c:\Users\SPCX\Desktop\github-repositories\aml-project2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [56]:
X = pd.read_csv("data/x_train.txt", sep=" ", header=None)
y = pd.read_csv("data/y_train.txt", sep=" ", header=None)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1, stratify=y
)
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [57]:
y_train = y_train.values.ravel()
y_test = y_test.values.ravel()

In [58]:
w_neg = (0.4888 / (1 - 0.4888)) * ((1 - 0.2) / 0.2)
class_weights = {
    1: 1.0,
    0: w_neg,
}  # because there are 1955 1s and 2045 0s in the training set
# but we want to mimic 20/80 ratio of 1s to 0s in the test set

In [59]:
clf = LogisticRegression(
    penalty="l2",
    solver="saga",
    class_weight=class_weights,
    random_state=1,
    max_iter=5000,
)
clf.fit(X_train, y_train)

LogisticRegression(class_weight={0: 3.8247261345852896, 1: 1.0}, max_iter=5000,
                   random_state=1, solver='saga')

In [60]:
# recall on test set
y_pred = clf.predict(X_test)
recall = recall_score(y_test, y_pred, pos_label=1)
print(f"Recall on test set: {recall:.4f}")

Recall on test set: 0.2270


In [ ]:
def business_scorer(estimator, X_val, y_val):
    proba = estimator.predict_proba(X_val)[:, 1]
    n_val = len(proba)
    top_obs = max(1, int(np.floor(0.2 * n_val)))
    top_indices = np.argsort(proba)[::-1][:top_obs]
    tp = np.sum(y_val[top_indices] == 1)
    k_features = X_val.shape[1]
    feature_cost = 200
    return ((tp / top_obs) * 10000 - feature_cost * k_features), tp, top_obs

In [105]:
def percent_correct_among_top20percent(estimator, X_val, y_val):
    proba = estimator.predict_proba(X_val)[:, 1]
    n_val = len(proba)
    top_obs = int(np.floor(0.2 * n_val))
    top_indices = np.argsort(proba)[::-1][:top_obs]
    tp = np.sum(y_val[top_indices] == 1)
    return tp / top_obs

In [ ]:
param_grid = {
    "n_estimators": [3000, 5000, 7000],
    "subsample": [0.2, 0.5, 0.7, 1.0],
    "reg_alpha": [0.0, 0.01, 0.05, 0.1, 0.5, 1.0],
    "reg_lambda": [0.0, 0.01, 0.05, 0.1, 0.5, 1.0],
    "q": [90, 95, 96, 97, 98, 99],
}

clf = XGBClassifier(
    eval_metric=percent_correct_among_top20percent,
    random_state=1,
    n_jobs=20,
    n_estimators=10000,
    max_depth=2,
    subsample=0.5,
    scale_pos_weight=1 / w_neg,
    reg_alpha=0.05,
    reg_lambda=0.01,
)
clf.fit(X_train, y_train, verbose=False)
gain = clf.get_booster().get_score(importance_type="gain")
q = 90
threshhold = np.percentile(list(gain.values()), q)
best_idx = np.where(np.array(list(gain.values())) > threshhold)[0]
clf.fit(X_train[:, best_idx], y_train, verbose=False)
y_pred = clf.predict(X_test[:, best_idx])
results = business_scorer(clf, X_test[:, best_idx], y_test)
print(f"Percent correct among top 20%: {results[1] / results[2]:.4f}")
print(f"Business score: {results[0]}")

Recall on test set: 0.5726
Found 143 true positives out of 200 top_obs.
So percent correct among top 20%: 0.7150
Business score: -2850.0


In [ ]:
mlflow.xgboost.autolog()


def objective(trial):
    n_estimators = trial.suggest_categorical("n_estimators", [3000, 5000, 7000])
    subsample_ = trial.suggest_float("subsample", 0.2, 1.0, step=0.01)
    reg_alpha = trial.suggest_float("reg_alpha", 0.0, 1.0, step=0.01)
    reg_lambda = trial.suggest_float("reg_lambda", 0.0, 1.0, step=0.01)
    q_percent = trial.suggest_float("q_percent", 90, 99.9, step=0.1)
    with mlflow.start_run(nested=True):
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("subsample", subsample_)
        mlflow.log_param("reg_alpha", reg_alpha)
        mlflow.log_param("reg_lambda", reg_lambda)
        mlflow.log_param("q_percent", q_percent)
        clf = XGBClassifier(
            eval_metric=percent_correct_among_top20percent,
            random_state=1,
            n_jobs=1,
            n_estimators=n_estimators,
            max_depth=2,
            subsample=subsample_,
            scale_pos_weight=(1.0 / w_neg),
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
        )
        clf.fit(X_train, y_train, verbose=False)
        booster = clf.get_booster()
        gain_dict = booster.get_score(importance_type="gain")
        if len(gain_dict) == 0:
            mlflow.log_metric("num_selected_features", 0)
            mlflow.log_metric("business_score", 0.0)
            return 0.0
        all_gains = np.array(list(gain_dict.values()))
        threshhold = np.percentile(all_gains, q_percent)
        selected_indices = np.where(np.array(list(gain.values())) > threshhold)[0]
        mlflow.log_metric("num_selected_features", len(selected_indices))
        mlflow.log_param("selected_features", str(selected_indices.tolist()))
        if len(selected_indices) == 0:
            mlflow.log_metric("business_score", 0.0)
            return 0.0
        X_train_sel = X_train[:, selected_indices]
        X_test_sel = X_test[:, selected_indices]
        clf_sel = XGBClassifier(
            eval_metric=percent_correct_among_top20percent,
            random_state=1,
            n_jobs=1,
            n_estimators=n_estimators,
            max_depth=2,
            subsample=subsample_,
            scale_pos_weight=(1.0 / w_neg),
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
        )
        clf_sel.fit(X_train_sel, y_train, verbose=False)
        business_score, n_corr_top20, total_top20 = business_scorer(
            clf_sel, X_test_sel, y_test
        )
        perc_in_top20 = n_corr_top20 / total_top20
        mlflow.log_metric("business_score", business_score)
        mlflow.log_metric("recall_top20", perc_in_top20)
        return business_score

2025/06/01 13:20:59 WARNING mlflow.utils.autologging_utils: MLflow xgboost autologging is known to be compatible with 1.4.2 <= xgboost <= 3.0.0, but the installed version is 3.0.2. If you encounter errors during autologging, try upgrading / downgrading xgboost to a compatible version, or try upgrading MLflow.


In [117]:
import warnings

warnings.filterwarnings("ignore")

study = optuna.create_study(direction="maximize", study_name="xgb_business_opt")
study.optimize(objective, timeout=21600)  # 6 hours
print("Optuna best business_score:", study.best_value)
print("Optuna best parameters:")
for key, val in study.best_params.items():
    print(f"    {key}: {val}")
best_params = study.best_params

2025/06/01 13:51:32 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during xgboost autologging: The following failures occurred while performing one or more logging operations: [MlflowException('Failed to perform one or more operations on the run with ID d69ae6b9c6bc4f48a0f5fefa148f12fc. Failed operations: [MlflowException("Changing param values is not allowed. Param with key=\'custom_metric\' was already logged with value=\'<function _metric_decorator.<locals>.inner at 0x000002CE16503380>\' for run ID=\'d69ae6b9c6bc4f48a0f5fefa148f12fc\'. Attempted logging new value \'<function _metric_decorator.<locals>.inner at 0x000002CE190D76A0>\'.")]')]
[I 2025-06-01 13:51:35,052] Trial 55 finished with value: 7500.0 and parameters: {'n_estimators': 3000, 'subsample': 0.21000000000000002, 'reg_alpha': 0.97, 'reg_lambda': 0.03, 'q_percent': 94.3}. Best is trial 17 with value: 7650.0.
2025/06/01 13:52:06 WARNING mlflow.utils.autologging_utils: Encountered unexpected error durin

Optuna best business_score: 7850.000000000001
Optuna best parameters:
    n_estimators: 7000
    subsample: 0.34
    reg_alpha: 0.1
    reg_lambda: 0.02
    q_percent: 98.9


In [119]:
# turn off autologging to avoid duplicate runs
mlflow.xgboost.autolog(disable=True)

In [136]:
best_model = XGBClassifier(
    eval_metric=percent_correct_among_top20percent,
    random_state=1,
    n_jobs=1,
    n_estimators=best_params["n_estimators"],
    max_depth=2,
    subsample=best_params["subsample"],
    scale_pos_weight=(1.0 / w_neg),
    reg_alpha=best_params["reg_alpha"],
    reg_lambda=best_params["reg_lambda"],
)
best_model.fit(X_train[:, [2]], y_train, verbose=False)
y_pred = best_model.predict(X_test[:, [2]])
results = business_scorer(best_model, X_test[:, [2]], y_test)
print(f"Percent correct among top 20%: {results[1] / results[2]:.4f}")
print(f"Business score: {results[0]}")

Percent correct among top 20%: 0.8050
Business score: 7850.000000000001


In [140]:
best_model = XGBClassifier(
    eval_metric=percent_correct_among_top20percent,
    random_state=1,
    n_jobs=1,
    n_estimators=best_params["n_estimators"],
    max_depth=2,
    subsample=best_params["subsample"],
    scale_pos_weight=(1.0 / w_neg),
    reg_alpha=best_params["reg_alpha"],
    reg_lambda=best_params["reg_lambda"],
)
best_model.fit(X.values[:, [2]], y, verbose=False)
X_production = pd.read_csv("data/x_test.txt", sep=" ", header=None)
print(f"Shape of production data: {X_production.shape}")
X_production = scaler.transform(X_production)
y_pred_proba = best_model.predict_proba(X_production[:, [2]])[:, 1]
top_1000_indices = np.argsort(y_pred_proba)[::-1][:1000]
with open("vars_xgb_final.txt", "w") as file:
    for num in [2]:  # only using the feature at index 2
        file.write(f"{num}\n")
with open("obs_xgb_final.txt", "w") as file:
    for i, num in enumerate(top_1000_indices):
        file.write(f"{i} {num}\n")

Shape of production data: (5000, 500)
